# Phase U — Capability suite (predictive-coding substrate)

One microcircuit, **one rule** `ΔW = η·Π·ε·φ(μ)`, **one objective** (free
energy) on an arbitrary, self-wiring graph.  Each section verifies a
load-bearing claim of `plan_unification.md` §9 against the **new API only**
— no legacy modules, no MJX, fully CPU-runnable.

## 0. Setup (Colab only — skip if running locally)

In [1]:
!pip -q install "jax[cuda12]" equinox numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 7.5 MB/s eta 0:00:00


In [2]:
import os, sys, subprocess
REPO = 'https://github.com/MateuszSieczka/Neuro_MVP'   # use the Phase-U branch/merge
DEST = '/content/Neuro_MVP'
if not os.path.isdir(DEST):
    subprocess.check_call(['git', 'clone', '-q', REPO, DEST])
os.chdir(DEST)
if DEST not in sys.path:
    sys.path.insert(0, DEST)
import jax
print('CWD =', os.getcwd(), '| jax', jax.__version__, '| devices', jax.devices())

CWD = /content/Neuro_MVP | jax 0.7.2 | devices [CudaDevice(id=0)]


## 1. Imports

In [3]:
import jax, jax.numpy as jnp

from core.backend import make_key
from core.pc_module import (
    init_pc_net_params, init_pc_net_state, pc_free_energy,
    pc_feedforward, pc_relax, pc_clamp_bottom, pc_fixed_prediction_grads, _phi,
)
from core.pc_graph import (
    init_pc_graph_params, init_pc_graph_state,
    pc_graph_clamp, pc_graph_relax, pc_graph_step, graph_free_energy,
    init_region_graph, REGION_INDEX,
)
from core.pc_brain import (
    init_pc_brain, pc_brain_cognitive_step, pc_brain_learn_forward, pc_brain_act,
)
from core.pc_active import (
    set_action_prior, pc_act_infer, pc_act_learn_forward, efe_select,
)
from core.pc_structural import (
    init_sparse_masks, pc_structural_learn, pc_structural_step, active_count,
)

verdicts = {}
print('jax', jax.__version__, '| devices', jax.devices())

jax 0.7.2 | devices [CudaDevice(id=0)]


## U.1a — Inference is relaxation: it descends free energy

`pc_relax` performs `μ̇ = −∂F/∂μ`.  Free energy must fall monotonically to
a fixed point — the dynamical core the legacy code never had (§9.1).

In [4]:
p = init_pc_net_params((4, 16, 16, 6), act='tanh', eta_mu=0.1, n_relax=200)
s = init_pc_net_state(make_key(0), p)
k = jax.random.split(make_key(1), 2)
inp = jax.random.normal(k[0], (6,))
ff = pc_feedforward(s, p, inp)
target = ff.mu[0] + 0.3 * jax.random.normal(k[1], (4,))
clamped = pc_clamp_bottom(ff, target)

Fs = [float(pc_free_energy(pc_relax(clamped, p, clamp_bottom=True, n_steps=n), p))
      for n in range(0, 201, 20)]
print('F over relaxation:', [round(f, 3) for f in Fs])
ok = Fs[-1] < Fs[0] * 0.7 and all(Fs[i + 1] <= Fs[i] + 1e-5 for i in range(len(Fs) - 1))
verdicts['U.1a relaxation descends F'] = ok
print('PASS' if ok else 'FAIL')

F over relaxation: [0.069, 0.041, 0.04, 0.04, 0.04, 0.04, 0.04, 0.04, 0.04, 0.04, 0.04]
PASS


## U.1b — The local rule == backprop (THE GATE, §9.2)

Under the fixed-prediction equilibrium the purely local Hebbian gradient
reproduces `jax.grad` of the equivalent feedforward net to cosine ≈ 1.0 —
backprop-grade credit assignment with no backprop.

In [5]:
p = init_pc_net_params((3, 12, 12, 5), act='tanh', eta_mu=0.1, n_relax=400)
s = init_pc_net_state(make_key(2), p)
inp = jax.random.normal(make_key(3), (5,))
target = jax.random.normal(make_key(4), (3,)) * 0.3

def ff_loss(w):
    a = inp
    for l in range(p.n_layers - 1, -1, -1):
        a = w[l] @ _phi(p.act, a)
    return 0.5 * jnp.sum((a - target) ** 2)

bp = jax.grad(ff_loss)(list(s.weights))
pc = pc_fixed_prediction_grads(s, p, inp, target)
af = jnp.concatenate([g.ravel() for g in pc])
bf = jnp.concatenate([g.ravel() for g in bp])
cos = float(af @ bf / (jnp.linalg.norm(af) * jnp.linalg.norm(bf)))
print('cosine(PC fixed-prediction grad, backprop grad) =', round(cos, 4))
ok = cos > 0.99
verdicts['U.1b PC == backprop'] = ok
print('PASS' if ok else 'FAIL')

cosine(PC fixed-prediction grad, backprop grad) = 1.0
PASS


## U.2 / §9.4 — One rule learns through a deep chain

A 4-edge generative chain; the single rule assigns credit all the way to
the top — the depth that node-perturbation could not scale to.

In [6]:
sizes = (3, 16, 16, 16, 6)
L = len(sizes) - 1
edges = tuple((l + 1, l) for l in range(L))          # higher predicts lower
p = init_pc_graph_params(sizes, edges, act='tanh', eta_mu=0.1, eta_w=5e-2, n_relax=40)
s = init_pc_graph_state(make_key(2), p)
top = p.n_nodes - 1
inp = jax.random.normal(make_key(3), (sizes[-1],))
target = jax.random.normal(make_key(4), (sizes[0],)) * 0.5

def chain_out(st):
    a = inp
    for l in range(L - 1, -1, -1):
        a = st.weights[l] @ _phi(p.act, a)
    return a

loss0 = float(0.5 * jnp.sum((chain_out(s) - target) ** 2))
for _ in range(300):
    s = pc_graph_step(s, p, {top: inp, 0: target}, n_steps=40).state
lossN = float(0.5 * jnp.sum((chain_out(s) - target) ** 2))
print(f'deep-chain loss {loss0:.4f} -> {lossN:.4f}')
ok = lossN < loss0 * 0.2
verdicts['U.2 deep credit, one rule'] = ok
print('PASS' if ok else 'FAIL')

deep-chain loss 0.4728 -> 0.0000
PASS


## §9.5 — Arbitrary topology (multi-parent + cycle) relaxes

No feedforward order required: a node with two parents and a 1↔2 cycle
still relaxes to finite, non-increasing free energy (Salvatori 2022).

In [7]:
sizes = (6, 8, 8, 5)
edges = ((1, 0), (2, 0), (3, 1), (1, 2), (2, 1))     # multi-parent + cycle
p = init_pc_graph_params(sizes, edges, eta_mu=0.05, n_relax=100)
s = init_pc_graph_state(make_key(4), p)
obs = jax.random.normal(make_key(5), (sizes[0],))
c = pc_graph_clamp(s, {0: obs})
f0 = float(graph_free_energy(c, p))
fN = float(graph_free_energy(pc_graph_relax(c, p, clamp=(0,), n_steps=100), p))
print(f'cyclic-graph FE {f0:.4f} -> {fN:.4f} | finite={bool(jnp.isfinite(fN))}')
ok = bool(jnp.isfinite(fN)) and fN <= f0 + 1e-5
verdicts['9.5 arbitrary topology'] = ok
print('PASS' if ok else 'FAIL')

cyclic-graph FE 3.0994 -> 0.9340 | finite=True
PASS


## U.2 big-bang — every region is a node of ONE graph

`init_region_graph` instantiates cortex / world-model / value / policy /
motor / cerebellum / hippocampus as nodes of a single graph, run by the
one rule.  One clamp→relax→learn cycle, no per-region rule.

In [8]:
p, s = init_region_graph(make_key(6), eta_mu=0.05, eta_w=1e-2, n_relax=30)
print('regions (nodes):', p.n_nodes, '| generative edges:', p.n_edges)
s_idx = REGION_INDEX['sensory']
obs = jax.random.normal(make_key(7), (p.node_sizes[s_idx],))
out = pc_graph_step(s, p, {s_idx: obs}, n_steps=30)
moved = sum(float(jnp.sum(jnp.abs(a - b))) for a, b in zip(out.state.weights, s.weights))
finite = bool(jnp.isfinite(out.free_energy)) and all(bool(jnp.all(jnp.isfinite(w))) for w in out.state.weights)
print(f'cycle FE={float(out.free_energy):.4f} | Δweights={moved:.4f}')
ok = finite and moved > 0
verdicts['U.2 region graph, one rule'] = ok
print('PASS' if ok else 'FAIL')

regions (nodes): 10 | generative edges: 13
cycle FE=0.9086 | Δweights=0.3760
PASS


## U.3 — Cognitive cycle = relaxation; perception learns

`pc_brain_cognitive_step` is clamp→relax→read motor→learn, no hand-coded
region sequence.  Repeated exposure lowers free energy (perception learns
to predict its afferent).  Also exercises the neuromodulatory precision
hook (ACh-like sensory gain) and the epistemic info-gain read-out.

In [9]:
params, state = init_pc_brain(make_key(0), sensory_size=8, motor_size=3,
                              eta_mu=0.05, eta_w=2e-2, n_relax=30)
sensory = jax.random.normal(make_key(1), (params.sensory_dim,))
fe0 = float(pc_brain_cognitive_step(state, params, sensory, learn=False).free_energy)
for _ in range(200):
    state = pc_brain_cognitive_step(state, params, sensory).state
probe = pc_brain_cognitive_step(state, params, sensory, learn=False)
feN = float(probe.free_energy)
print(f'perceptual FE {fe0:.4f} -> {feN:.4f} | epistemic info-gain={float(probe.epistemic):.4f}')

base = pc_brain_cognitive_step(state, params, sensory, learn=False)
gated = pc_brain_cognitive_step(state, params, sensory, learn=False,
                                precision_gains={REGION_INDEX['sensory']: 10.0})
shift = float(jnp.sum(jnp.abs(gated.joint_command - base.joint_command)))
print(f'precision-gain (neuromod) motor shift = {shift:.5f}')
ok = feN < fe0 * 0.7 and shift > 1e-5 and bool(jnp.isfinite(probe.epistemic))
verdicts['U.3 perception + neuromodulation'] = ok
print('PASS' if ok else 'FAIL')

AttributeError: 'PCBrainOutput' object has no attribute 'epistemic'

## U.5 — Babble → reach, NO REINFORCE

A forward model learned from random commands (one rule) lets action be
*inferred* (clamp preferred outcome, relax, read motor) — the capability
M1 node-perturbation existed for, now from local PC inference (§9.6).

In [ ]:
d = 4
p = init_pc_graph_params((d, d), ((1, 0),), act='linear', eta_mu=0.3, eta_w=0.05, n_relax=60)
s = init_pc_graph_state(make_key(0), p)
A = jax.random.normal(make_key(1), (d, d)) * 0.6 + jnp.eye(d)   # synthetic body / plant
qstar = jax.random.normal(make_key(2), (d,))
s = set_action_prior(s, 1)                                      # action carries a flat prior

for kk in jax.random.split(make_key(3), 300):                  # canonical babbling
    cmd = jax.random.normal(kk, (d,)) * 0.5
    s = pc_act_learn_forward(s, p, 1, 0, cmd, A @ cmd, update_precision=False)
fm_err = float(jnp.linalg.norm(s.weights[0] - A))

out = pc_act_infer(s, p, 1, 0, qstar, n_steps=150)            # reach by inference
realised = A @ out.command
rel = float(jnp.linalg.norm(qstar - realised) / jnp.linalg.norm(qstar))
print(f'forward-model error={fm_err:.3f} | reach rel error={rel:.3f}')

# Same principle through the full brain API — command is goal-sensitive.
bp, bs = init_pc_brain(make_key(7), sensory_size=6, motor_size=3,
                       eta_mu=0.2, eta_w=3e-2, n_relax=30)
Bm = jax.random.normal(make_key(8), (6, 3)) * 0.5
for kk in jax.random.split(make_key(9), 200):
    c = jax.random.normal(kk, (3,)) * 0.5
    bs = pc_brain_learn_forward(bs, bp, c, Bm @ c)
ca = pc_brain_act(bs, bp, jnp.ones(6) * 0.5, n_relax=80)
cb = pc_brain_act(bs, bp, -jnp.ones(6) * 0.5, n_relax=80)
print('brain goal-sensitivity |cmd_a - cmd_b| =', round(float(jnp.sum(jnp.abs(ca - cb))), 4))

ok = rel < 0.1 and float(jnp.sum(jnp.abs(ca - cb))) > 1e-3
verdicts['U.5 babble→reach, no REINFORCE'] = ok
print('PASS' if ok else 'FAIL')

## U.5 — Action selection = argmin expected free energy

One equation unifies exploitation and curiosity: with β=0 the choice is
greedy-pragmatic, with large β it is epistemic (curiosity-driven).

In [ ]:
pragmatic = jnp.array([1.0, 0.5, 0.2])     # policy 0 best for the goal
epistemic = jnp.array([0.0, 0.0, 2.0])     # policy 2 most informative
greedy = efe_select(pragmatic, epistemic, epistemic_weight=0.0)
curious = efe_select(pragmatic, epistemic, epistemic_weight=5.0)
print('greedy pick:', int(greedy.index), '| curious pick:', int(curious.index))
ok = int(greedy.index) == 0 and int(curious.index) == 2
verdicts['U.5 argmin EFE'] = ok
print('PASS' if ok else 'FAIL')

## U.4b — Self-wiring: structure follows free energy

The graph grows edges where they reduce FE (same gradient as learning),
prunes weak ones, and caps density (homeostasis).  Self-wiring beats a
frozen-sparse graph that only adjusts existing weights (§9.5).

In [ ]:
p = init_pc_graph_params((4, 12, 6), ((1, 0), (2, 1)), act='tanh',
                          eta_mu=0.1, eta_w=0.05, n_relax=40)
k1, k2, k3, k4 = jax.random.split(make_key(0), 4)
s0 = init_pc_graph_state(k1, p)
s0, masks0 = init_sparse_masks(s0, p, k2, density=0.1)
inp = jax.random.normal(k3, (6,))
target = jax.random.normal(k4, (4,)) * 0.5
top = 2

def ffl(st):
    a = st.weights[1] @ _phi('tanh', inp)
    a = st.weights[0] @ _phi('tanh', a)
    return float(0.5 * jnp.sum((a - target) ** 2))

sb = s0                                                   # frozen-sparse baseline
for _ in range(150):
    c = pc_graph_clamp(sb, {top: inp, 0: target})
    r = pc_graph_relax(c, p, clamp=(0, top), n_steps=40)
    sb = pc_structural_learn(r, p, masks0)
loss_base = ffl(sb)

sg, mg = s0, masks0                                       # self-wiring
for _ in range(150):
    out = pc_structural_step(sg, p, mg, {top: inp, 0: target},
                             n_steps=40, spawn_threshold=0.03, max_active_frac=0.6)
    sg, mg = out.state, out.masks
loss_grow = ffl(sg)

print(f'frozen-sparse: loss={loss_base:.4f} ({int(active_count(masks0))} syn)'
      f' | self-wired: loss={loss_grow:.4f} ({int(active_count(mg))} syn)')
ok = int(active_count(mg)) > int(active_count(masks0)) and loss_grow < loss_base * 0.5
verdicts['U.4b self-wiring'] = ok
print('PASS' if ok else 'FAIL')

## Verdict matrix

In [ ]:
print('=== PHASE-U CAPABILITY VERDICTS ===')
for k, v in verdicts.items():
    print(f'  [{"PASS" if v else "FAIL"}] {k}')
n_pass = sum(bool(v) for v in verdicts.values())
print(f'\n{n_pass}/{len(verdicts)} capabilities verified')